# LLFC via CKA — Without Alignment

This notebook tests whether two matching tickets satisfy **Linear Feature Connectivity ** at the penultimate layer, measured via CKA — without any explicit weight-space alignment (e.g., permutation matching).

## Idea

Given two models θ_A and θ_B, for each α ∈ [0, 1]:

1. **Weight interpolation**: build the intermediate model θ(α) = (1−α)·θ_A + α·θ_B
2. **Extract** penultimate-layer features from θ(α) on the test set → **h_α**
3. **Linearly interpolate** the endpoint features: **h_lin(α)** = (1−α)·h_A + α·h_B
4. **Compute** CKA(h_α, h_lin(α))

If the two models are **LLFC-compatible**, the weight-interpolated features h_α should be well-approximated by the feature-interpolation h_lin(α), so **CKA should remain high and flat across α** (especially near α=0.5).

CKA is invariant to orthogonal transformations, so this acts as an alignment-free probe.

## Hypothesis space

| LMC | LLFC_CKA | Interpretation |
|-----|----------|----------------|
| ✓   | ✓        | Consistent with theory (LLFC ⟹ LMC) |
| ✓   | ✗        | Interesting: weight-space linearity doesn't carry over to feature-space |
| ✗   | ✗        | Consistent (LLFC is stronger than LMC) |

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import numpy as np
import matplotlib.pyplot as plt
import copy

from models import BigCNN, StudentCNN
from lth_utils import get_prunable_layers
from training import eval_epoch
from mnist1d_dataset import get_mnist1d_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_loader, test_loader = get_mnist1d_loaders()
criterion = nn.CrossEntropyLoss()

## Core helper functions

In [ ]:
def extract_features(model, layer_path: str, dataloader, device) -> torch.Tensor:
    """
    Run the model on all batches and collect activations at `layer_path`.
    Returns a tensor of shape (N, d) — one row per sample.
    """
    model.eval()
    model.to(device)
    collected = []

    def hook(_, __, output):
        collected.append(output.detach().cpu())

    modules = dict(model.named_modules())
    handle = modules[layer_path].register_forward_hook(hook)

    with torch.no_grad():
        for batch in dataloader:
            x = batch[0] if isinstance(batch, (list, tuple)) else batch
            model(x.to(device))

    handle.remove()
    feats = torch.cat(collected, dim=0)          # (N, d)
    return feats.view(feats.size(0), -1)         # flatten spatial dims if any


def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Unbiased linear CKA between two feature matrices X, Y of shape (N, d).
    Mean-centers each matrix before computing HSIC.
    """
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    hsic  = torch.norm(X.T @ Y, p='fro') ** 2
    norm_x = torch.norm(X.T @ X, p='fro')
    norm_y = torch.norm(Y.T @ Y, p='fro')

    if norm_x == 0 or norm_y == 0:
        return float('nan')
    return (hsic / (norm_x * norm_y)).item()


def recompute_bn_stats(model, data_loader, device, num_batches: int = 50):
    """Re-estimate BatchNorm running stats for the interpolated model."""
    model.train()
    with torch.no_grad():
        for i, (x, _) in enumerate(data_loader):
            if i >= num_batches:
                break
            model(x.to(device))
    model.eval()

## Loading helpers

In [ ]:
def load_pruned_bigcnn(ckpt_path: str, device) -> BigCNN:
    """Load a BigCNN from a checkpoint that was saved with pruning reparametrization."""
    model = BigCNN().to(device)
    for m, p in get_prunable_layers(model):
        prune.identity(m, p)  # re-apply the pruning structure before loading
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model


def load_student(ckpt_path: str, device) -> StudentCNN:
    model = StudentCNN().to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()
    return model

In [ ]:
def interpolate_pruned_bigcnn(model_a: BigCNN, model_b: BigCNN, alpha: float, device) -> BigCNN:
    """
    θ(α) = (1-α)·θ_A + α·θ_B  — interpolates the full state dict
    (weights + pruning masks are blended, matching the approach used in
     Ticket_and_Students_Connectivity.ipynb).
    """
    interp = BigCNN().to(device)
    for m, p in get_prunable_layers(interp):
        prune.identity(m, p)

    sd = {}
    for (k, va), (_, vb) in zip(model_a.state_dict().items(), model_b.state_dict().items()):
        sd[k] = (1 - alpha) * va + alpha * vb
    interp.load_state_dict(sd)
    interp.eval()
    return interp


def interpolate_student(model_a: StudentCNN, model_b: StudentCNN, alpha: float, device) -> StudentCNN:
    interp = StudentCNN().to(device)
    sd = {}
    for (k, va), (_, vb) in zip(model_a.state_dict().items(), model_b.state_dict().items()):
        sd[k] = (1 - alpha) * va + alpha * vb
    interp.load_state_dict(sd)
    interp.eval()
    return interp

## LLFC_CKA experiment loop

Single function that runs the experiment for any model pair and returns the results.

In [ ]:
def run_llfc_cka_experiment(
    model_a,
    model_b,
    interpolate_fn,     # (model_a, model_b, alpha, device) -> interpolated model
    layer_path: str,    # e.g. "classifier.1"
    alphas,
    dataloader,
    train_loader,
    device,
    label: str = "",
):
    """
    For each α:
      1. Build θ(α) = (1-α)·θ_A + α·θ_B
      2. Extract h_α  = penultimate features of θ(α)
      3. Compute h_lin(α) = (1-α)·h_A + α·h_B
      4. CKA(h_α, h_lin(α))  — the LLFC_CKA score

    Also records:
      - CKA(h_α, h_A)  — how far weight-interpolated features drift from endpoint A
      - error rate of θ(α)  — for LMC context

    Returns a dict with arrays indexed by alpha.
    """
    print(f"\n{'='*60}")
    print(f"  LLFC_CKA experiment: {label}")
    print(f"  Layer: {layer_path}  |  #alphas: {len(alphas)}")
    print(f"{'='*60}")

    # Pre-extract endpoint features (done once, reused for all α)
    print("Extracting endpoint features h_A and h_B ...")
    h_A = extract_features(model_a, layer_path, dataloader, device)  # (N, d)
    h_B = extract_features(model_b, layer_path, dataloader, device)  # (N, d)
    print(f"  h_A shape: {h_A.shape}  |  h_B shape: {h_B.shape}")

    results = dict(alphas=list(alphas), llfc_cka=[], cka_vs_A=[], errors=[])

    for alpha in alphas:
        # 1. Build interpolated model
        theta_alpha = interpolate_fn(model_a, model_b, alpha, device)

        # Re-estimate BN stats so the interpolated model is well-calibrated
        recompute_bn_stats(theta_alpha, train_loader, device)

        # 2. Extract h_α
        h_alpha = extract_features(theta_alpha, layer_path, dataloader, device)

        # 3. Linearly interpolate endpoint features
        h_lin_alpha = (1 - alpha) * h_A + alpha * h_B

        # 4. LLFC_CKA: does weight-interpolated model match feature-interpolation?
        cka_llfc = linear_cka(h_alpha, h_lin_alpha)

        # Auxiliary: how far does h_α drift from h_A?
        cka_a = linear_cka(h_alpha, h_A)

        # Error rate for LMC context
        _, acc = eval_epoch(theta_alpha, dataloader, criterion, device=device)
        error = 1.0 - acc

        results['llfc_cka'].append(cka_llfc)
        results['cka_vs_A'].append(cka_a)
        results['errors'].append(error)

        print(f"  α={alpha:.2f}  |  LLFC_CKA={cka_llfc:.4f}  "
              f"|  CKA(h_α,h_A)={cka_a:.4f}  |  Error={error:.4f}")

    endpoint_error = max(results['errors'][0], results['errors'][-1])
    barrier = max(results['errors']) - endpoint_error
    print(f"\n  Error barrier (LMC): {barrier:.4f}")
    print(f"  Mean LLFC_CKA:       {np.mean(results['llfc_cka']):.4f}")
    print(f"  LLFC_CKA at α=0.5:   {results['llfc_cka'][len(alphas)//2]:.4f}")
    results['error_barrier'] = barrier
    return results

In [ ]:
def plot_llfc_cka(results, label: str, axs=None, show: bool = True):
    """
    Two-panel figure:
      Left : LLFC_CKA(α)  and  CKA(h_α, h_A)(α)
      Right: Error rate(α)  — LMC landscape
    """
    alphas  = results['alphas']
    llfc    = results['llfc_cka']
    cka_a   = results['cka_vs_A']
    errors  = results['errors']
    barrier = results['error_barrier']

    standalone = axs is None
    if standalone:
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f"LLFC_CKA experiment — {label}", fontsize=13, fontweight='bold')

    # --- Left: CKA curves ---
    ax = axs[0]
    ax.plot(alphas, llfc,  marker='o', lw=2, label='CKA(h_α , h_lin(α))  [LLFC_CKA]', color='tab:green')
    ax.plot(alphas, cka_a, marker='s', lw=2, linestyle='--', label='CKA(h_α , h_A)', color='tab:orange')
    ax.axhline(1.0, color='gray', lw=0.8, linestyle=':')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('α', fontsize=12)
    ax.set_ylabel('CKA', fontsize=12)
    ax.set_title(f'{label}\nLLFC_CKA along interpolation path', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    for x, y in zip(alphas, llfc):
        ax.annotate(f'{y:.2f}', xy=(x, y), xytext=(0, 7),
                    textcoords='offset points', ha='center', fontsize=8, color='tab:green')

    # --- Right: Error rate (LMC landscape) ---
    ax2 = axs[1]
    ax2.plot(alphas, errors, marker='o', lw=2, color='tab:blue', label=f'Error rate (barrier={barrier:.3f})')
    ax2.axhline(errors[0],  color='gray', lw=0.8, linestyle=':', label='endpoint error')
    ax2.axhline(errors[-1], color='gray', lw=0.8, linestyle=':')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, max(errors) * 1.3 + 0.01)
    ax2.set_xlabel('α', fontsize=12)
    ax2.set_ylabel('Test error rate', fontsize=12)
    ax2.set_title(f'{label}\nLMC landscape (error barrier={barrier:.3f})', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    for x, y in zip(alphas, errors):
        ax2.annotate(f'{y:.3f}', xy=(x, y), xytext=(0, 7),
                     textcoords='offset points', ha='center', fontsize=8, color='tab:blue')

    if standalone:
        plt.tight_layout()
        plt.show()

---
## Experiment 1 — Matching Tickets

Expected outcome: **LMC ✓ → LLFC_CKA ?**

In [ ]:
matching_A = load_pruned_bigcnn("models/matching/matching_model_run0.pth", device)
matching_B = load_pruned_bigcnn("models/matching/matching_model_run1.pth", device)

_, acc_A = eval_epoch(matching_A, test_loader, criterion, device=device)
_, acc_B = eval_epoch(matching_B, test_loader, criterion, device=device)
print(f"Matching ticket A — test accuracy: {acc_A:.4f}")
print(f"Matching ticket B — test accuracy: {acc_B:.4f}")

In [ ]:
PENULTIMATE_BIGCNN = "classifier.1"   # Linear(128, 64) — output is the 64-dim representation

alphas = np.linspace(0.0, 1.0, 11)

results_matching = run_llfc_cka_experiment(
    model_a       = matching_A,
    model_b       = matching_B,
    interpolate_fn= interpolate_pruned_bigcnn,
    layer_path    = PENULTIMATE_BIGCNN,
    alphas        = alphas,
    dataloader    = test_loader,
    train_loader  = train_loader,
    device        = device,
    label         = "Matching Tickets",
)

In [ ]:
plot_llfc_cka(results_matching, label="Matching Tickets")

---
## Experiment 2 — Regular LTH Tickets

Expected outcome: **LMC ✗ → LLFC_CKA ✗** (consistent, since LLFC is stronger than LMC)

In [ ]:
ticket_A = load_pruned_bigcnn("models/lth/ticket_model_run0.pth", device)
ticket_B = load_pruned_bigcnn("models/lth/ticket_model_run1.pth", device)

_, acc_A = eval_epoch(ticket_A, test_loader, criterion, device=device)
_, acc_B = eval_epoch(ticket_B, test_loader, criterion, device=device)
print(f"LTH ticket A — test accuracy: {acc_A:.4f}")
print(f"LTH ticket B — test accuracy: {acc_B:.4f}")

In [ ]:
results_lth = run_llfc_cka_experiment(
    model_a       = ticket_A,
    model_b       = ticket_B,
    interpolate_fn= interpolate_pruned_bigcnn,
    layer_path    = PENULTIMATE_BIGCNN,
    alphas        = alphas,
    dataloader    = test_loader,
    train_loader  = train_loader,
    device        = device,
    label         = "LTH Tickets (independent)",
)

In [ ]:
plot_llfc_cka(results_lth, label="LTH Tickets (independent)")

---
## Experiment 3 — KD Students (independent training)

Expected outcome: **LMC ✗ → LLFC_CKA ✗**

In [ ]:
student_A = load_student("models/best_kd_student_0.pth", device)
student_B = load_student("models/best_kd_student_1.pth", device)

_, acc_A = eval_epoch(student_A, test_loader, criterion, device=device)
_, acc_B = eval_epoch(student_B, test_loader, criterion, device=device)
print(f"KD student A — test accuracy: {acc_A:.4f}")
print(f"KD student B — test accuracy: {acc_B:.4f}")

In [ ]:
PENULTIMATE_STUDENT = "classifier.1"  # Linear(320, 19) — output is the 19-dim representation

results_students = run_llfc_cka_experiment(
    model_a       = student_A,
    model_b       = student_B,
    interpolate_fn= interpolate_student,
    layer_path    = PENULTIMATE_STUDENT,
    alphas        = alphas,
    dataloader    = test_loader,
    train_loader  = train_loader,
    device        = device,
    label         = "KD Students (independent)",
)

In [ ]:
plot_llfc_cka(results_students, label="KD Students (independent)")

---
## Summary: all three cases on one figure

Top row: LLFC_CKA(α) — the main new metric.  
Bottom row: error rate(α) — LMC landscape for reference.

In [ ]:
all_results = [
    (results_matching, "Matching Tickets"),
    (results_lth,      "LTH Tickets (indep.)"),
    (results_students, "KD Students (indep.)"),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("LLFC_CKA experiment — summary", fontsize=14, fontweight='bold')

for col, (res, lbl) in enumerate(all_results):
    # top row: CKA curves
    ax_top = axes[0, col]
    ax_top.plot(res['alphas'], res['llfc_cka'], marker='o', lw=2,
                color='tab:green', label='LLFC_CKA')
    ax_top.plot(res['alphas'], res['cka_vs_A'], marker='s', lw=2,
                linestyle='--', color='tab:orange', label='CKA(h_α, h_A)')
    ax_top.set_ylim(0, 1.05)
    ax_top.set_xlim(0, 1)
    ax_top.set_title(lbl, fontsize=11)
    ax_top.set_ylabel('CKA' if col == 0 else '')
    ax_top.grid(True, alpha=0.3)
    ax_top.legend(fontsize=8)

    # bottom row: error rate
    ax_bot = axes[1, col]
    ax_bot.plot(res['alphas'], res['errors'], marker='o', lw=2,
                color='tab:blue', label=f"barrier={res['error_barrier']:.3f}")
    ax_bot.set_ylim(0, max(res['errors']) * 1.4 + 0.01)
    ax_bot.set_xlim(0, 1)
    ax_bot.set_xlabel('α')
    ax_bot.set_ylabel('Test error rate' if col == 0 else '')
    ax_bot.grid(True, alpha=0.3)
    ax_bot.legend(fontsize=8)

axes[0, 0].set_title("Matching Tickets\n(LMC ✓ expected)", fontsize=10)
axes[0, 1].set_title("LTH Tickets (indep.)\n(LMC ✗ expected)", fontsize=10)
axes[0, 2].set_title("KD Students (indep.)\n(LMC ✗ expected)", fontsize=10)
axes[0, 0].set_ylabel('CKA', fontsize=11)
axes[1, 0].set_ylabel('Test error rate', fontsize=11)

plt.tight_layout()
plt.savefig("llfc_cka_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to llfc_cka_summary.png")

---
## Numerical summary table

In [ ]:
print(f"{'Model pair':<30} {'LMC barrier':>12} {'mean LLFC_CKA':>14} {'LLFC_CKA @0.5':>14}")
print("-" * 72)
for res, lbl in all_results:
    mid = res['llfc_cka'][len(alphas) // 2]
    print(f"{lbl:<30} {res['error_barrier']:>12.4f} "
          f"{np.mean(res['llfc_cka']):>14.4f} {mid:>14.4f}")